In [30]:
#环境变量设置
import os
os.environ['OPENAI_API_KEY'] = 'sk-proj-VZY6S96CMCmZpZvwUhsHT3BlbkFJG52BaSQYIHestC8t9e6Q'
os.environ["http_proxy"] = "http://localhost:15732"
os.environ["https_proxy"] = "http://localhost:15732"
command ="../../../config/command.wav"

In [14]:
#hello world
#简单的测试一下openai的接口
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print(completion.choices[0].message)



ChatCompletionMessage(content="QUERY='Red coke can'", refusal=None, role='assistant', function_call=None, tool_calls=None)


In [31]:
#语音转文字
from openai import OpenAI
client = OpenAI()
audio_file= open(command, "rb")
transcription = client.audio.transcriptions.create(
  model="whisper-1", 
  file=audio_file
)
print(transcription.text)

请说出你的称号 请帮我抓一只小黄鸭


In [4]:
#准备prompt
single_en_prompt = """
            你是一个机器人助手，你有一个机械臂，你的任务是帮助用户执行抓取物体和放置物体的指令，你的下游是 CLIP，它会负责识别你的指令，请你尽可能考虑自己的输出是否能够让下游更好的理解。
            你需要从输入的一段文本中提取关键点：
            第一、需要你观察图片，识别出哪个物体是被抓取的，要能清晰识别被动语态。
            第二、需要从输入的文本中整理出要抓取物体的各种特征，包括颜色、形状、种类等
            第三，注意动词的不同表达方式，比如抓和拿等。
            第四，输出的结果只能是颜色（颜色需要通过对图片分析给出）+名词的形式，比如Red coke can，而不是Red coke can in a bottle。
            第五、输出结果一定要按照以下格式展示：QUERY。（QUERY，表示要抓取目标的名称)
            第六、如果图片中不存在目标物体，输出QUERY='None'。
            补充一点，输出的QUERY请换成英文
            示例：
            指令：帮我抓一个豆奶。
            以下是你要输出的内容：
            QUERY='Green bean milk'

            指令：把可乐拿给我
            以下是你要输出的内容：
            QUERY='Red coke can '

            指令：把桌子上的雪碧放到蓝色的框里。
            以下是你要输出的内容：
            QUERY='Green Sprite can'

            指令：拿一下鸭子。
            以下是你要输出的内容：
            QUERY='Yellow duck'

            指令：把万用表递给我。
            以下是你要输出的内容：
            QUERY='Red surgical multimeter'

            指令：把AD钙奶递给我。
            以下是你要输出的内容：
            QUERY='Green AD Milk'

            指令：递给我咖啡。
            以下是你要输出的内容：
            QUERY='Brown Nestle coffee'

            指令：把柠檬茶放到篮子里。
            以下是你要输出的内容：
            QUERY='Yellow lemon tea'

            指令：把红牛递给我吧。
            以下是你要输出的内容：
            QUERY='Red Bull'

            指令：帮我拿一下卷尺。
            以下是你要输出的内容：
            QUERY='Yellow tape measure'

            指令：你能把芬达放到我面前吗？
            以下是你要输出的内容：
            QUERY='Orange Fanta can'

            指令：INSERT TASK HERE
            """
def insert_task_into_prompt(task, prompt_base, insert_index="INSERT TASK HERE"):
    full_prompt = prompt_base.replace(insert_index, task)
    return full_prompt



In [ ]:
#测试一下
prompt=insert_task_into_prompt("把可乐递给我吧。", single_en_prompt)

In [34]:
#进行对话
client = OpenAI()
completion = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print(completion.choices[0].message.content)
#进行切片操作，提取出' '中的内容
query = completion.choices[0].message.content.split("'")[1]
print(query)

QUERY='Red coke can'
Red coke can


In [36]:
#图片+文字对话
import base64
import requests

prompt=insert_task_into_prompt("把可乐给我", single_en_prompt)
# OpenAI API Key
api_key = "sk-proj-VZY6S96CMCmZpZvwUhsHT3BlbkFJG52BaSQYIHestC8t9e6Q"

# Function to encode the image
def encode_image(image_path):
  with open(image_path, "rb") as image_file:
    return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "../../../config/over.jpeg"

# Getting the base64 string
base64_image = encode_image(image_path)

headers = {
  "Content-Type": "application/json",
  "Authorization": f"Bearer {api_key}"
}

payload = {
  "model": "chatgpt-4o-latest",
  "messages": [
    {
      "role": "user",
      "content": [
        {
          "type": "text",
          "text": prompt
        },
        {
          "type": "image_url",
          "image_url": {
            "url": f"data:image/jpeg;base64,{base64_image}"
          }
        }
      ]
    }
  ],
  "max_tokens": 300
}

response = requests.post("https://api.openai.com/v1/chat/completions", headers=headers, json=payload)

print(response.json())
#进行切片操作，提取出' '中的内容
result=response.json()["choices"][0]["message"]["content"].split("'")[1]
print(result)


{'id': 'chatcmpl-A3dJYX57SzkdjMkE9MMJoV7QST3jV', 'object': 'chat.completion', 'created': 1725429472, 'model': 'chatgpt-4o-latest', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "QUERY='Red coke can'", 'refusal': None}, 'logprobs': None, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 1015, 'completion_tokens': 6, 'total_tokens': 1021}, 'system_fingerprint': 'fp_ed47180004'}
Red coke can
